In [ ]:
# Clone the CUT repository
!git clone https://github.com/taesungp/contrastive-unpaired-translation.git
%cd contrastive-unpaired-translation

In [ ]:
# Install dependencies
# Note: Colab already has torch and torchvision; we only install CUT-specific packages
!pip install dominate visdom GPUtil packaging

In [ ]:
import os
import zipfile
import shutil
import re
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Mount Google Drive to access dataset and save checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
zip_path = "/content/drive/MyDrive/Diego/Disney_GAN/dataset_GAN_faces.zip"  
extract_to = "/content/temp_dataset"

# CUT needs a structure with trainA and trainB folders
dataset_root = "./datasets/disney_cut"

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print(f"ZIP extracted to: {extract_to}")

# Building trainA and trainB from train subfolder
os.makedirs(f"{dataset_root}/trainA", exist_ok=True)
os.makedirs(f"{dataset_root}/trainB", exist_ok=True)

# Copy train/live_action to trainA
live_source = os.path.join(extract_to, "train", "live_action")
for item in os.listdir(live_source):
    s = os.path.join(live_source, item)
    d = os.path.join(f"{dataset_root}/trainA", item)
    if os.path.isfile(s):
        shutil.copy2(s, d)

# Copy train/cartoon to trainB
cartoon_source = os.path.join(extract_to, "train", "cartoon")
for item in os.listdir(cartoon_source):
    s = os.path.join(cartoon_source, item)
    d = os.path.join(f"{dataset_root}/trainB", item)
    if os.path.isfile(s):
        shutil.copy2(s, d)

print(f"trainA (live-action): {len(os.listdir(f'{dataset_root}/trainA'))} images")
print(f"trainB (cartoon): {len(os.listdir(f'{dataset_root}/trainB'))} images")

In [ ]:
# Download pretrained Generator weights
print("Downloading pretrained models...")
!mkdir -p ./checkpoints
!wget http://efrosgans.eecs.berkeley.edu/CUT/pretrained_models.tar
!tar -xf pretrained_models.tar
!mv horse2zebra_* cat2dog_* cityscapes_* ./checkpoints/ 2>/dev/null || true

print("Pretrained models extracted to ./checkpoints/")


In [ ]:

%cd /content/contrastive-unpaired-translation

!rm -rf ./checkpoints/disney_FastCUT
# Fix argparse bug in CUT repo (choices should be a list, not a string)
!sed -i "s/choices='(CUT, cut, FastCUT, fastcut)'/choices=['CUT', 'cut', 'FastCUT', 'fastcut']/g" models/cut_model.py

print("Initialize all networks (1 epoch)")
# Train for 1 epoch to initialize all networks (G, D, F)
!python train.py --dataroot ./datasets/disney_cut --name disney_FastCUT --model cut --CUT_mode FastCUT --dataset_mode unaligned --direction AtoB --gpu_ids 0 --batch_size 32 --n_epochs 1 --n_epochs_decay 0 --display_id -1 --no_html --save_epoch_freq 1 2>&1 | tee training_stage1.log

print("Inject pretrained Generator weights")
# Replace the randomly initialized Generator with pretrained weights
!cp ./checkpoints/horse2zebra_fastcut_pretrained/latest_net_G.pth ./checkpoints/disney_FastCUT/latest_net_G.pth


print("3: Resume training with pretrained G")
# Continue training with pretrained G, keeping D and F as they are
!python train.py --dataroot ./datasets/disney_cut --name disney_FastCUT --model cut --CUT_mode FastCUT --dataset_mode unaligned --direction AtoB --gpu_ids 0 --batch_size 32 --n_epochs 100 --n_epochs_decay 50 --epoch_count 2 --display_id -1 --no_html --continue_train --epoch latest 2>&1 | tee training_stage3.log


## Visualize Training Losses

Plot the training losses from the log file to monitor training progress.

In [ ]:


def plot_cut_losses(log_paths=["training_stage1.log", "training_stage3.log"], save_path="/content/cut_losses.png"):
    epochs, iters_list = [], []
    G_GAN, D_real, D_fake, NCE = [], [], [], []
    
    # Parse all log files (stage 1 + stage 3)
    for log_path in log_paths:
        if not os.path.exists(log_path):
            continue
        with open(log_path, 'r') as f:
            for line in f:
                # Match CUT loss format: (epoch: X, iters: Y, ...) G_GAN: ... D_real: ... D_fake: ... NCE: ...
                m = re.search(
                    r'epoch:\s*(\d+).*?iters:\s*(\d+).*?'
                    r'G_GAN:\s*([\d.]+).*?'
                    r'D_real:\s*([\d.]+).*?'
                    r'D_fake:\s*([\d.]+).*?'
                    r'NCE:\s*([\d.]+)',
                    line
                )
                if m:
                    epochs.append(int(m.group(1)))
                    iters_list.append(int(m.group(2)))
                    G_GAN.append(float(m.group(3)))
                    D_real.append(float(m.group(4)))
                    D_fake.append(float(m.group(5)))
                    NCE.append(float(m.group(6)))
    
    
    # 2x2 figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1: NCE Loss 
    axes[0, 0].plot(epochs, NCE, color='#ff6b6b', linewidth=0.8, alpha=0.7)
    axes[0, 0].set_title('NCE Loss (Contrastive)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2: Generator GAN Loss
    axes[0, 1].plot(epochs, G_GAN, color='#4a9eff', linewidth=0.8, alpha=0.7)
    axes[0, 1].set_title('Generator GAN Loss', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3: Discriminator Losses
    axes[1, 0].plot(epochs, D_real, color='#51cf66', linewidth=0.8, alpha=0.7, label='D_real')
    axes[1, 0].plot(epochs, D_fake, color='#ffd43b', linewidth=0.8, alpha=0.7, label='D_fake')
    axes[1, 0].set_title('Discriminator Losses', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4: All combined and normalized
    def normalize(arr):
        arr = np.array(arr)
        return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    
    axes[1, 1].plot(epochs, normalize(NCE), color='#ff6b6b', linewidth=0.8, alpha=0.7, label='NCE (norm)')
    axes[1, 1].plot(epochs, normalize(G_GAN), color='#4a9eff', linewidth=0.8, alpha=0.7, label='G_GAN (norm)')
    axes[1, 1].plot(epochs, normalize(D_real), color='#51cf66', linewidth=0.8, alpha=0.7, label='D_real (norm)')
    axes[1, 1].plot(epochs, normalize(D_fake), color='#ffd43b', linewidth=0.8, alpha=0.7, label='D_fake (norm)')
    axes[1, 1].set_title('All Losses (Normalized)', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Normalized Loss')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    for ax in axes.flat:
        ax.spines[['top', 'right']].set_visible(False)
    
    plt.suptitle('FastCUT Training Losses', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

plot_cut_losses(["training_stage1.log", "training_stage3.log"])

## Save Checkpoints to Google Drive

After training completes, copy the final checkpoint to your Google Drive for safekeeping and later inference.

The generator weights are stored in `latest_net_G.pth`.

In [ ]:
import shutil

# Save model
models_dir = "/content/drive/MyDrive/Diego/Disney_GAN/CUT_checkpoints/"
os.makedirs(models_dir, exist_ok=True)

checkpoint_source = "./checkpoints/disney_FastCUT/"
checkpoint_dest = os.path.join(models_dir, "disney_FastCUT")

shutil.copytree(checkpoint_source, checkpoint_dest, dirs_exist_ok=True)

print(f"Checkpoints saved to {checkpoint_dest}")
print(f"weights: {checkpoint_dest}/latest_net_G.pth")